# Taintwatch benchmark report

Reproduces the tables in `results/report.md` from `results/records.csv`.
Generate that file first with `python -m taintwatch.evaluation`.
Intervals are 95% Wilson score intervals.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from taintwatch.evaluation import (
    attack_breakdown,
    benign_breakdown,
    compare_to_pivot,
    read_records,
    summarize,
)

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent
records = read_records(ROOT / 'results' / 'records.csv')
summaries = summarize(records)
names = [s.defense for s in summaries]
print(len(records), 'runs;', ', '.join(names))

## Overall

In [ ]:
print(f"{'defense':<20}{'attack success (95% CI)':<30}{'benign utility':>16}")
for s in summaries:
    print(f'{s.defense:<20}{s.attack_success.fmt():<30}{s.benign_utility.percent():>16}')

In [ ]:
def bars(title, rates, color):
    labels = [name for name, _ in rates]
    values = [r.value * 100 for _, r in rates]
    low = [max(0.0, (r.value - r.interval()[0]) * 100) for _, r in rates]
    high = [max(0.0, (r.interval()[1] - r.value) * 100) for _, r in rates]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(labels) + 1.5))
    ax.barh(labels, values, xerr=[low, high], color=color, capsize=3)
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel('percent')
    ax.set_title(title)
    fig.tight_layout()
    plt.show()


bars('Attack success (lower is better)',
     [(s.defense, s.attack_success) for s in summaries], '#8a2a1b')
bars('Benign utility (higher is better)',
     [(s.defense, s.benign_utility) for s in summaries], '#1f5c4d')

An observed 0% is not a zero rate: with 600 attacks the upper end of the interval is about 0.6%.
The attack goals match the sinks the default policy guards, so the taintwatch result is partly
by construction.

## Breakdowns

In [ ]:
for key in ('goal', 'carrier', 'style'):
    print('Attack success by', key)
    for name in names:
        groups = attack_breakdown(records, name, key)
        cells = ', '.join(f'{k}={r.percent()}' for k, r in groups.items())
        print(f'  {name:<20}{cells}')
    print()

In [ ]:
print('Benign utility by family')
for name in names:
    groups = benign_breakdown(records, name)
    cells = ', '.join(f'{k.split("/")[-1]}={r.percent()}' for k, r in groups.items())
    print(f'  {name:<20}{cells}')

## Paired comparison against taintwatch

In [ ]:
for p in compare_to_pivot(records, names, 'taintwatch'):
    print(f'{p.a:<20} only-it {p.only_a:>4}  only-taintwatch {p.only_b:>4}  p={p.p_value:.2e}')